In [1]:
import napari
import scipy.ndimage as ndi
import cv2
import skimage as ski
import glob
import numpy as np
import nd2
import pandas as pd
import plotly.express as px
from roifile import ImagejRoi

In [2]:
viewer = napari.Viewer()

In [3]:
def rotate_xy(inp, angle, interp=True):
    shape = np.array(inp.shape[1:])
    shape = shape[::-1]
    center = shape / 2
    center
    rotation_matrix = cv2.getRotationMatrix2D(center, angle, 1.0)
    if interp:
        output = np.array([cv2.warpAffine(inp[i], rotation_matrix, shape) for i in range(inp.shape[0])])
    else:
        output = np.array([cv2.warpAffine(inp[i], rotation_matrix, shape, flags=cv2.INTER_NEAREST) for i in range(inp.shape[0])])
    return output

def rotate_xz(inp, angle, interp=True):
    shape = np.array(inp.shape)[[2,0]]
    center = np.array(shape) / 2
    rotation_matrix = cv2.getRotationMatrix2D(center, angle, 1.0)
    if interp:
        output = np.array([cv2.warpAffine(inp[:,i], rotation_matrix, shape) for i in range(inp.shape[1])])
    else:
        output = np.array([cv2.warpAffine(inp[:,i], rotation_matrix, shape, flags=cv2.INTER_NEAREST) for i in range(inp.shape[1])])
    output = np.swapaxes(output, 0, 1)
    return output



In [12]:
fnames = glob.glob('*/*/SeanCrops/*.zip')

In [13]:
fname = fnames[0]
fname

'Cen11g1r\\rep1\\SeanCrops\\Image001-1_points.zip'

In [14]:
def get_pos(rois):
    x =[]
    y =[]
    z = []
    positions = []
    for roi in rois:
        y.append(roi.subpixel_coordinates[0][1])
        x.append(roi.subpixel_coordinates[0][0])
        z.append(1.87 * roi.z_position)
        positions.append(np.floor((roi.position-1))/4*1.87)
    x = np.array(x)
    y = np.array(y)
    z = np.array(z)
    positions = np.array(positions)
    if np.sum(np.abs(z))==0:
        z = positions
    return np.array([np.array(z), np.array(y), np.array(x)]).T

In [15]:
def rotate_img(img, pts, do_rot=False):
    cimg = img.copy()
    center = np.mean(pts, axis=0)
    img_center = np.array(cimg.shape) / 2
    shifted = np.roll(cimg, shift=(img_center-center).astype(int), axis=(0,1,2))  # Does roll so it wraps around, kind of annoying
    dP = pts[0] - pts[1]
    R = np.linalg.norm(dP)
    phi = np.arctan2(dP[1], dP[2]) # Comes back as radians
    theta = np.arccos(dP[0]/R)

    if do_rot:
        rotated = rotate_xy(shifted, phi * 180 / np.pi)
        z_rotated = rotate_xz(rotated, 90 - theta * 180 / np.pi)
    else:
        z_rotated = shifted
    return z_rotated, center, phi, theta

In [16]:
def process_fname(fname, display=False, viewer=None):
    rois = ImagejRoi.fromfile(fname)
    img = ski.io.imread(fname.replace('_points.zip', '.tif'))
    img = ski.transform.rescale(img, (1.87,1,1,1), preserve_range=True)
    all_pts = get_pos(rois)

    pts = all_pts[0:2]

    # Make rotated image (not necessary but fun)
    chimg = []
    for idx in range(img.shape[-1]):
        tmp, center, phi, theta = rotate_img(img[:,:,:,idx], pts, do_rot=display)
        chimg.append(tmp)
    chimg = np.stack(chimg, axis=-1)

    img_center = np.array(img.shape[:3]) / 2
    shift = img_center-center
    shifted_pts = all_pts + shift

    z_theta = theta - np.pi/2
    phi_rotation_matrix = np.array([
        [1, 0, 0],
        [0, np.cos(phi), -np.sin(phi)],
        [0, np.sin(phi), np.cos(phi)]
    ])
    theta_rotation_matrix = np.array([
        [np.cos(z_theta), 0, np.sin(z_theta)],
        [0, 1, 0],
        [-np.sin(z_theta), 0, np.cos(z_theta)]
    ])

    phi_rotated_pts = shifted_pts - img_center
    phi_rotated_pts = phi_rotated_pts@phi_rotation_matrix.T
    phi_rotated_pts = phi_rotated_pts + np.array(img.shape[:3])/2

    theta_rotated_pts = phi_rotated_pts - np.array(img.shape[:3])/2
    theta_rotated_pts = theta_rotated_pts@theta_rotation_matrix.T
    theta_rotated_pts = theta_rotated_pts + np.array(img.shape[:3])/2

    final_pts = (theta_rotated_pts - np.array(img.shape[:3])/2) * .107

    if display:
        viewer.add_image(img, channel_axis=-1)
        viewer.add_points(all_pts)
        viewer.add_image(chimg, channel_axis=-1)
        viewer.add_points(theta_rotated_pts)
    
    cdf = pd.DataFrame(final_pts, columns=['z', 'y', 'x'])
    cdf['Point'] = [0,1,2,3,4,5]
    cdf['fname'] = fname
    
    return cdf


In [9]:
viewer.layers.clear()
final_pts = process_fname(fnames[6], display=True, viewer=viewer)

In [17]:
import dask

df = []
for fname in fnames:
    df.append(dask.delayed(process_fname)(fname, display=False))
df = dask.compute(*df)
df = pd.concat(df, ignore_index=True)
df

,z,y,x,Point,fname
0,1.520561e-15,0.000000,5.418176,0,Cen11g1r\rep1\SeanCrops\Image001-1_points.zip
1,-1.520561e-15,0.000000,-5.418176,1,Cen11g1r\rep1\SeanCrops\Image001-1_points.zip
2,-9.870978e-01,-4.567088,-0.379792,2,Cen11g1r\rep1\SeanCrops\Image001-1_points.zip
3,2.078794e+00,1.188093,-2.070877,3,Cen11g1r\rep1\SeanCrops\Image001-1_points.zip
4,8.355490e-01,-2.306936,-0.937438,4,Cen11g1r\rep1\SeanCrops\Image001-1_points.zip
...,...,...,...,...,...
3667,-7.602807e-16,0.000000,-7.818854,1,Cen7g1r\rep3\SeanCrops\image020-9_points.zip
3668,-1.486385e+00,-1.537640,-1.059869,2,Cen7g1r\rep3\SeanCrops\image020-9_points.zip
3669,-8.571064e-01,4.210521,-0.706204,3,Cen7g1r\rep3\SeanCrops\image020-9_points.zip
3670,1.258351e+00,3.899908,-1.901640,4,Cen7g1r\rep3\SeanCrops\image020-9_points.zip


In [18]:
df.to_csv('Results.csv')

# Analyze Results

In [19]:
df = pd.read_csv('Results.csv')

In [20]:
df

,Unnamed: 0,z,y,x,Point,fname
0,0,1.520561e-15,0.000000,5.418176,0,Cen11g1r\rep1\SeanCrops\Image001-1_points.zip
1,1,-1.520561e-15,0.000000,-5.418176,1,Cen11g1r\rep1\SeanCrops\Image001-1_points.zip
2,2,-9.870978e-01,-4.567088,-0.379792,2,Cen11g1r\rep1\SeanCrops\Image001-1_points.zip
3,3,2.078794e+00,1.188093,-2.070877,3,Cen11g1r\rep1\SeanCrops\Image001-1_points.zip
4,4,8.355490e-01,-2.306936,-0.937438,4,Cen11g1r\rep1\SeanCrops\Image001-1_points.zip
...,...,...,...,...,...,...
3667,3667,-7.602807e-16,0.000000,-7.818854,1,Cen7g1r\rep3\SeanCrops\image020-9_points.zip
3668,3668,-1.486385e+00,-1.537640,-1.059869,2,Cen7g1r\rep3\SeanCrops\image020-9_points.zip
3669,3669,-8.571064e-01,4.210521,-0.706204,3,Cen7g1r\rep3\SeanCrops\image020-9_points.zip
3670,3670,1.258351e+00,3.899908,-1.901640,4,Cen7g1r\rep3\SeanCrops\image020-9_points.zip


In [21]:
df['r'] = np.sqrt(df['x']**2 + df['y']**2 + df['z']**2)

In [22]:
df['centromere'] = df['fname'].str.split('\\').str[0]
df['rep'] = df['fname'].str.split('\\').str[1]
df

,Unnamed: 0,z,y,x,Point,fname,r,centromere,rep
0,0,1.520561e-15,0.000000,5.418176,0,Cen11g1r\rep1\SeanCrops\Image001-1_points.zip,5.418176,Cen11g1r,rep1
1,1,-1.520561e-15,0.000000,-5.418176,1,Cen11g1r\rep1\SeanCrops\Image001-1_points.zip,5.418176,Cen11g1r,rep1
2,2,-9.870978e-01,-4.567088,-0.379792,2,Cen11g1r\rep1\SeanCrops\Image001-1_points.zip,4.687953,Cen11g1r,rep1
3,3,2.078794e+00,1.188093,-2.070877,3,Cen11g1r\rep1\SeanCrops\Image001-1_points.zip,3.165672,Cen11g1r,rep1
4,4,8.355490e-01,-2.306936,-0.937438,4,Cen11g1r\rep1\SeanCrops\Image001-1_points.zip,2.626573,Cen11g1r,rep1
...,...,...,...,...,...,...,...,...,...
3667,3667,-7.602807e-16,0.000000,-7.818854,1,Cen7g1r\rep3\SeanCrops\image020-9_points.zip,7.818854,Cen7g1r,rep3
3668,3668,-1.486385e+00,-1.537640,-1.059869,2,Cen7g1r\rep3\SeanCrops\image020-9_points.zip,2.386839,Cen7g1r,rep3
3669,3669,-8.571064e-01,4.210521,-0.706204,3,Cen7g1r\rep3\SeanCrops\image020-9_points.zip,4.354519,Cen7g1r,rep3
3670,3670,1.258351e+00,3.899908,-1.901640,4,Cen7g1r\rep3\SeanCrops\image020-9_points.zip,4.517628,Cen7g1r,rep3


In [23]:
px.box(df[df['Point'].isin([0,1])], x='rep', y='r', color='Point', width=800, points='all', range_y=[0,10], facet_col='centromere')

In [24]:
df['ax'] = np.abs(df['x'])
df['ay'] = np.abs(df['y'])
df['az'] = np.abs(df['z'])
melted_df = df.melt(id_vars=['fname', 'Point', 'centromere', 'rep'], value_vars=['ax', 'ay', 'az', 'r'], var_name='coordinate', value_name='value')

In [25]:
mapper = {0:'AuroraA', 1:'AuroraB', 2:'Cen1 Dim', 3:'Cen1 Bright', 4:'Cen7-11 Dim', 5:'Cen7-11 Bright'}
melted_df['Position'] = melted_df['Point'].map(mapper)


In [26]:
f = px.box(melted_df[melted_df['Point'].isin([0,1])], x='rep', facet_row='centromere', y='value', color='Position', facet_col='coordinate', width=1000, height=800, points='all', range_y=[-1,10], title='Distance From Metaphase Plate Center (Absolute Values)<br>X Along Aurora Axis')
f.write_html('AuroraDistances.html')
f

In [27]:
f = px.box(melted_df[melted_df['Point'].isin([2,3])], facet_row='centromere', x='rep', y='value', height=800, color='Position', facet_col='coordinate', width=1000, points='all', range_y=[-1,10], title='Distance From Metaphase Plate Center (Absolute Values)<br>X Along Aurora Axis')
f.write_html('Cen1Distances.html')
f

In [28]:
melted_df

,fname,Point,centromere,rep,coordinate,value,Position
0,Cen11g1r\rep1\SeanCrops\Image001-1_points.zip,0,Cen11g1r,rep1,ax,5.418176,AuroraA
1,Cen11g1r\rep1\SeanCrops\Image001-1_points.zip,1,Cen11g1r,rep1,ax,5.418176,AuroraB
2,Cen11g1r\rep1\SeanCrops\Image001-1_points.zip,2,Cen11g1r,rep1,ax,0.379792,Cen1 Dim
3,Cen11g1r\rep1\SeanCrops\Image001-1_points.zip,3,Cen11g1r,rep1,ax,2.070877,Cen1 Bright
4,Cen11g1r\rep1\SeanCrops\Image001-1_points.zip,4,Cen11g1r,rep1,ax,0.937438,Cen7-11 Dim
...,...,...,...,...,...,...,...
14683,Cen7g1r\rep3\SeanCrops\image020-9_points.zip,1,Cen7g1r,rep3,r,7.818854,AuroraB
14684,Cen7g1r\rep3\SeanCrops\image020-9_points.zip,2,Cen7g1r,rep3,r,2.386839,Cen1 Dim
14685,Cen7g1r\rep3\SeanCrops\image020-9_points.zip,3,Cen7g1r,rep3,r,4.354519,Cen1 Bright
14686,Cen7g1r\rep3\SeanCrops\image020-9_points.zip,4,Cen7g1r,rep3,r,4.517628,Cen7-11 Dim


In [29]:
f = px.box(melted_df[melted_df['Point'].isin([4,5])], facet_row='centromere', y='value', x='rep', height=800, color='Position', facet_col='coordinate', width=1000, points='all', range_y=[-1,10], 
           title='Distance From Metaphase Plate Center (Absolute Values)<br>X Along Aurora Axis', hover_data=['fname'])
f.write_html('Cen7_11Distances.html')
f

In [30]:
import plotly.graph_objects as go
f=go.FigureWidget(
    px.box(melted_df[melted_df['Point'].isin([4,5])], facet_row='centromere', y='value', x='rep', height=800, color='Position', facet_col='coordinate', width=1000, points='all', range_y=[-1,10], 
           title='Distance From Metaphase Plate Center (Absolute Values)<br>X Along Aurora Axis', hover_data=['fname'])
    )
                   
def click_fn(trace, points, state):
    
    if (len(points.point_inds)>0):
        fname = f.data[points.trace_index]['customdata'][points.point_inds[-1]][0]
        viewer.layers.clear()
        process_fname(fname, display=True, viewer=viewer)
        viewer.layers[0].name = fname


for data in f.data:
    data.on_click(click_fn)
f

FigureWidget({
    'data': [{'alignmentgroup': 'True',
              'boxpoints': 'all',
              'customdata': array([['Cen11g1r\\rep1\\SeanCrops\\Image001-1_points.zip'],
                                   ['Cen11g1r\\rep1\\SeanCrops\\Image001-2_points.zip'],
                                   ['Cen11g1r\\rep1\\SeanCrops\\Image002-1_points.zip'],
                                   ['Cen11g1r\\rep1\\SeanCrops\\Image003-1_points.zip'],
                                   ['Cen11g1r\\rep1\\SeanCrops\\Image003-2_points.zip'],
                                   ['Cen11g1r\\rep1\\SeanCrops\\Image003-3_points.zip'],
                                   ['Cen11g1r\\rep1\\SeanCrops\\Image004-1_points.zip'],
                                   ['Cen11g1r\\rep1\\SeanCrops\\Image004-2_points.zip'],
                                   ['Cen11g1r\\rep1\\SeanCrops\\Image005-1_points.zip'],
                                   ['Cen11g1r\\rep1\\SeanCrops\\Image006-1_points.zip'],
                     

In [31]:
melted_df.to_csv('AggregatedDistances.csv', index=False)